# T Industries - Genie Agents (idempotent)
Creates or updates two Genie Agents through the Genie REST API, then links **T Sales & Ops** to the *T Commercial Overview* dashboard:

| Genie Agent | Data | Typical questions |
| --- | --- | --- |
| **T Sales & Ops** | metric view `t_sales_metrics`, `gold_price_benchmark`, `gold_product_margin`, `gold_inventory_position` (row-filtered), `gold_ticket_insights`, dims, function `identify_product_photo` (all in schema `t_hardware`) | Which SKUs are priced >10% above competitors? Top customers by margin in BC last quarter. What is this product (photo) and how does our price compare? |
| **T Docs & SOPs** | `silver_doc_chunks`, `gold_doc_summaries` (ai_parse_document / ai_extract output) | What is the warranty period on commercial closers? How do I install a surface closer? |

Every example SQL is executed first, so a broken example fails the run instead of silently teaching Genie bad SQL.

In [0]:
import json, uuid
from databricks.sdk import WorkspaceClient
dbutils.widgets.text("catalog", "mdk_demo_catalog")
dbutils.widgets.text("warehouse_id", "1f421abc4c14022f")
dbutils.widgets.text("dashboard_name", "T Commercial Overview")
C = dbutils.widgets.get("catalog"); WH = dbutils.widgets.get("warehouse_id")
G = S = f"{C}.t_hardware"     # single schema; table names carry the layer prefix (gold_, silver_)
w = WorkspaceClient(); api = w.api_client.do
ME = w.current_user.me().user_name

def ids(n):
    """Genie returns list items by ascending id - generate sorted ids so items keep the order we write them in."""
    return sorted(uuid.uuid4().hex for _ in range(n))

def check(sql):
    spark.sql(sql).limit(1).collect()     # fail fast on a broken example
    return sql

def space_json(tables, instructions, examples, samples, functions=(), benchmarks=()):
    ex_ids, s_ids, f_ids, b_ids = ids(len(examples)), ids(len(samples)), ids(len(functions)), ids(len(benchmarks))
    return json.dumps({
        "version": 2,
        "config": {"sample_questions": [{"id": i, "question": [q]} for i, q in zip(s_ids, samples)]},
        "data_sources": {"tables": [{"identifier": t} for t in sorted(tables)]},
        "instructions": {
            "text_instructions": [{"id": uuid.uuid4().hex, "content": [l + "\n" for l in instructions]}],
            "example_question_sqls": [{"id": i, "question": [q], "sql": [check(s)]} for i, (q, s) in zip(ex_ids, examples)],
            "sql_functions": [{"id": i, "identifier": f} for i, f in zip(f_ids, functions)],
        },
        "benchmarks": {"questions": [{"id": i, "question": [q], "answer": [{"format": "SQL", "content": [check(s)]}]} for i, (q, s) in zip(b_ids, benchmarks)]},
    })

def upsert_space(title, description, serialized):
    existing = [s for s in api("GET", "/api/2.0/genie/spaces").get("spaces", []) if s["title"] == title]
    body = {"title": title, "description": description, "warehouse_id": WH, "serialized_space": serialized}
    if existing:
        sid = existing[0]["space_id"]; api("PATCH", f"/api/2.0/genie/spaces/{sid}", body=body); print("updated", title, sid)
    else:
        sid = api("POST", "/api/2.0/genie/spaces", body={**body, "parent_path": f"/Users/{ME}"})["space_id"]; print("created", title, sid)
    return sid

In [0]:
sales_instructions = [
    "You answer sales, margin, pricing, inventory and service questions for T Industries, a Canadian distributor of door hardware (locks, levers, door closers, hinges, exit devices) and bathroom fixtures (faucets, grab bars, bath accessories). All money is CAD.",
    f"Use the metric view {G}.t_sales_metrics with MEASURE() for net sales, gross margin, gross margin %, units, ASP, backorder rate and customer counts. Never re-derive these from gold_sales_daily.",
    "Net sales = gross sales - discounts - returns, excluding freight and tax; cancelled orders are excluded (Finance rule). Backorder rate = backordered units / ordered units (Operations rule; target < 5%, > 10% for a category triggers a supply review).",
    "Regions are BC, ON and US (customer region). Warehouses are 'Delta BC' and 'Toronto'. 'Last quarter' means the previous complete calendar quarter: order_quarter = add_months(date_trunc('QUARTER', current_date()), -3).",
    f"Competitor pricing: {G}.gold_price_benchmark has one row per SKU. pct_gap_vs_min > 0.10 means T's list price is more than 10% above the cheapest competitor. Use {G}.gold_price_benchmark_by_competitor for per-competitor detail (NorthBuild Supply, Pacific Hardware Co, DoorPro Direct).",
    f"When a user mentions a photo or image file (e.g. site_photo_exit_device.png), call {G}.identify_product_photo('<file name>') - it reads the photo with a vision model and returns the SKU and its competitor price benchmark.",
    f"{G}.gold_inventory_position is row-filtered: users only see the warehouses they are entitled to. If a warehouse is missing from results, say the user may not have access to it.",
    "Show percentages as percentages with one decimal. Round currency to whole dollars unless asked for unit prices.",
]
sales_examples = [
    ("Which SKUs are priced more than 10% above competitors?",
     f"SELECT sku, product_name, category, t_list_price_cad, competitor_min_price_cad, cheapest_competitor, competitor_avg_price_cad, pct_gap_vs_min FROM {G}.gold_price_benchmark WHERE pct_gap_vs_min > 0.10 ORDER BY pct_gap_vs_min DESC"),
    ("Top customers by margin in BC last quarter",
     f"SELECT customer_name, segment, MEASURE(gross_margin) AS gross_margin, MEASURE(net_sales) AS net_sales, MEASURE(gross_margin_pct) AS gross_margin_pct FROM {G}.t_sales_metrics WHERE region = 'BC' AND order_quarter = add_months(date_trunc('QUARTER', current_date()), -3) GROUP BY ALL ORDER BY gross_margin DESC LIMIT 10"),
    ("Monthly net sales and gross margin % for the last 12 months",
     f"SELECT order_month, MEASURE(net_sales) AS net_sales, MEASURE(gross_margin_pct) AS gross_margin_pct FROM {G}.t_sales_metrics WHERE order_date >= add_months(date_trunc('MONTH', current_date()), -12) GROUP BY ALL ORDER BY order_month"),
    ("Backorder rate by warehouse and category this quarter",
     f"SELECT warehouse, category, MEASURE(backorder_rate) AS backorder_rate, MEASURE(units_backordered) AS units_backordered FROM {G}.t_sales_metrics WHERE order_quarter = date_trunc('QUARTER', current_date()) GROUP BY ALL ORDER BY backorder_rate DESC"),
    ("Which SKUs are below reorder point in Toronto?",
     f"SELECT sku, product_name, category, available_qty, on_order_qty, reorder_point, days_of_cover, stock_status FROM {G}.gold_inventory_position WHERE warehouse = 'Toronto' AND below_reorder_point ORDER BY days_of_cover NULLS FIRST"),
    ("Gross margin % by category over the last 12 months",
     f"SELECT category, MEASURE(net_sales) AS net_sales, MEASURE(gross_margin_pct) AS gross_margin_pct FROM {G}.t_sales_metrics WHERE order_date > add_months(current_date(), -12) GROUP BY ALL ORDER BY gross_margin_pct"),
    ("What are the top service ticket themes this quarter?",
     f"SELECT theme, count(*) AS tickets, count_if(is_open) AS open_tickets, round(avg(days_to_close), 1) AS avg_days_to_close FROM {G}.gold_ticket_insights WHERE created_date >= date_trunc('QUARTER', current_date()) GROUP BY theme ORDER BY tickets DESC"),
    ("What is this product in site_photo_exit_device.png, and how does our price compare to competitors?",
     f"SELECT * FROM {G}.identify_product_photo('site_photo_exit_device.png')"),
]
sales_samples = ["Which SKUs are priced more than 10% above competitors?", "Top customers by margin in BC last quarter",
                 "What is this product in site_photo_door_closer.png, and how does our price compare to competitors?",
                 "Why are Toronto door closer backorders so high?", "What are the top service ticket themes this quarter?"]
sales_benchmarks = [
    ("How many exit device SKUs are more than 10% above the cheapest competitor?", f"SELECT count(*) AS skus FROM {G}.gold_price_benchmark WHERE category = 'Exit Devices' AND pct_gap_vs_min > 0.10"),
    ("What was net revenue last year?", f"SELECT MEASURE(net_sales) AS net_sales FROM {G}.t_sales_metrics WHERE order_year = year(current_date()) - 1"),
    ("Which warehouse has the higher backorder rate?", f"SELECT warehouse, MEASURE(backorder_rate) AS backorder_rate FROM {G}.t_sales_metrics GROUP BY ALL ORDER BY backorder_rate DESC LIMIT 1"),
    ("What's our GM% on faucets?", f"SELECT MEASURE(gross_margin_pct) AS gross_margin_pct FROM {G}.t_sales_metrics WHERE category = 'Faucets'"),
    ("Who is the cheapest competitor most often?", f"SELECT cheapest_competitor, count(*) AS skus FROM {G}.gold_price_benchmark GROUP BY ALL ORDER BY skus DESC LIMIT 1"),
]
sales_tables = [f"{G}.t_sales_metrics", f"{G}.gold_price_benchmark", f"{G}.gold_price_benchmark_by_competitor", f"{G}.gold_product_margin",
                f"{G}.gold_inventory_position", f"{G}.gold_ticket_insights", f"{G}.gold_dim_product", f"{G}.gold_dim_customer"]
sales_id = upsert_space("T Sales & Ops",
    "TAI backend for sales, margin, competitor pricing, inventory and service questions at T Industries (door hardware and bathroom fixtures, BC / ON / US). Built on governed gold tables and the t_sales_metrics metric view.",
    space_json(sales_tables, sales_instructions, sales_examples, sales_samples, [f"{G}.identify_product_photo"], sales_benchmarks))

In [0]:
docs_instructions = [
    "You answer questions from T Industries' SharePoint documents (spec sheets, installation SOPs, warranty policy, returns/RMA SOP, pricing authority matrix, Canadian HR handbook excerpt, business-rule definitions) and HubSpot ticket attachments (quotes, site photos, portal screenshots).",
    f"Start with {G}.gold_doc_summaries (one row per document: summary, document_type, product_family, warranty_terms, install_steps, key_policy_points). For exact wording, search {S}.silver_doc_chunks.chunk_text with case-insensitive LIKE / ILIKE on 2-3 keywords and return doc_name, section, page and chunk_text.",
    "Always cite the document name (and section/page when available) in the answer. If the documents do not contain the answer, say so - do not guess.",
    "Product vocabulary: 'closer' = door closer; 'panic bar' / 'crash bar' = exit device; 'commercial' = ANSI/BHMA Grade 1; 'residential' = Grade 3. WP-01 Warranty Policy is the authoritative warranty source; spec sheets summarise it.",
]
docs_examples = [
    ("What is the warranty period on commercial closers?",
     f"SELECT doc_name, section, page, chunk_text FROM {S}.silver_doc_chunks WHERE chunk_text ILIKE '%closer%' AND chunk_text ILIKE '%warrant%' ORDER BY CASE WHEN doc_name ILIKE 'WP-01%' THEN 0 ELSE 1 END, doc_name"),
    ("How do I install a surface door closer?",
     f"SELECT doc_name, install_steps FROM {G}.gold_doc_summaries WHERE document_type = 'installation SOP' AND doc_name ILIKE '%door closer%'"),
    ("Which documents mention warranty terms?",
     f"SELECT doc_name, document_type, product_family, warranty_terms FROM {G}.gold_doc_summaries WHERE warranty_terms IS NOT NULL ORDER BY doc_name"),
    ("What is the restocking fee for returns?",
     f"SELECT doc_name, section, page, chunk_text FROM {S}.silver_doc_chunks WHERE chunk_text ILIKE '%restock%'"),
    ("How is net sales defined?",
     f"SELECT doc_name, chunk_text FROM {S}.silver_doc_chunks WHERE folder = 'Business Rules' AND chunk_text ILIKE '%net sales%'"),
    ("What attachments were uploaded on pricing dispute tickets?",
     f"SELECT ticket_id, doc_name, summary FROM {G}.gold_doc_summaries WHERE source = 'HubSpot' AND document_type = 'quote' ORDER BY ticket_id"),
]
docs_samples = ["What is the warranty period on commercial closers?", "How do I install a surface door closer?",
                "What is the restocking fee for returns?", "How many vacation days do new employees get?", "Who can approve a 35% discount off list?"]
docs_id = upsert_space("T Docs & SOPs",
    "TAI backend for product, installation, warranty, returns, pricing-policy and HR questions, grounded in parsed SharePoint documents and HubSpot attachments (ai_parse_document).",
    space_json([f"{S}.silver_doc_chunks", f"{G}.gold_doc_summaries"], docs_instructions, docs_examples, docs_samples))

In [0]:
DASH = dbutils.widgets.get("dashboard_name")
dash = [d for d in w.lakeview.list() if d.display_name and d.display_name.endswith(DASH)]
for d in dash:
    full = w.lakeview.get(d.dashboard_id)
    spec = json.loads(full.serialized_dashboard)
    if spec.get("uiSettings", {}).get("genieSpace", {}).get("overrideId") == sales_id:
        # Already linked via the bundle's .lvdash.json - don't touch the bundle-managed dashboard (avoids deploy drift)
        print("dashboard already linked:", d.display_name, "->", sales_id); continue
    # New workspace / new space id: link it, and put this id into src/dashboards/t_commercial_overview.lvdash.json
    spec.setdefault("uiSettings", {})["genieSpace"] = {"isEnabled": True, "overrideId": sales_id, "enablementMode": "ENABLED"}
    api("PATCH", f"/api/2.0/lakeview/dashboards/{d.dashboard_id}", body={"serialized_dashboard": json.dumps(spec)})
    try: w.lakeview.publish(d.dashboard_id, warehouse_id=WH, embed_credentials=True)
    except Exception as e: print("publish skipped:", e)
    print("linked dashboard", d.display_name, "->", sales_id, "- update uiSettings.genieSpace.overrideId in the .lvdash.json to match")
if not dash: print(f"dashboard '{DASH}' not found - deploy the bundle first")
dbutils.jobs.taskValues.set("sales_space_id", sales_id); dbutils.jobs.taskValues.set("docs_space_id", docs_id)
print(json.dumps({"T Sales & Ops": sales_id, "T Docs & SOPs": docs_id}, indent=1))